# ArtifactBench 1.2-rc2 — metadata quickstart

This notebook verifies the frozen public metadata, not detector accuracy or audio rights. It needs no GPU, audio files, model weights, or network access during execution. The eight-model benchmark and paper remain in preparation until their separate result gates pass.

The pinned manifest is shared with the dataset card and paper construction table. Native platform outputs can include edits or conditioned generation; they are not universally established as fully synthetic. Three extra transport views are verified same-recording MP3/AAC pairs, not three extra independent recordings. The superseded rc1 confused seven numbered Udio sibling outputs with transport variants and selected five wrong primary files; rc2 corrects these using full page-declared URLs, without detector-score selection.

In [ ]:
from collections import Counter
import json
import os
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'artifactbench/v12/verify_release.py').is_file())
sys.path.insert(0, str(ROOT))
from artifactbench.v12.verify_release import verify
assert Path(sys.modules[verify.__module__].__file__).resolve().is_relative_to(ROOT.resolve()), 'Verifier imported outside bundle root'

RELEASE = Path(os.environ.get('ARTIFACTBENCH_RELEASE', str(ROOT / 'out/v1.2_frozen_rc2_260905')))
EXPECTED_MANIFEST_SHA256 = 'feab7c4c3d037919fd784dc40e46199470088abde532c5181e34573f20dceefd'
expected_python = os.environ.get('ARTIFACTBENCH_NOTEBOOK_PYTHON')
if expected_python:
    assert Path(sys.executable).absolute() == Path(expected_python).absolute(), 'Unexpected notebook kernel executable'
expected_prefix = os.environ.get('ARTIFACTBENCH_NOTEBOOK_PREFIX')
if expected_prefix:
    assert Path(sys.prefix).resolve() == Path(expected_prefix).resolve(), 'Unexpected notebook kernel environment'
print('Metadata-only execution; Python', sys.version.split()[0])
print('Local verifier and requested kernel identity checks passed.')

## Verify identity and privacy boundaries

The verifier checks public-file SHA-256 hashes, unique entry IDs, source/label/partition totals, recording representatives, dependence clusters, and native-pair links. It rejects local path fields and raw lyric/prompt fields. This does **not** re-decode the original audio or establish redistribution permission.

In [ ]:
verification = verify(RELEASE)
assert verification['manifest_sha256'] == EXPECTED_MANIFEST_SHA256, 'Not the pinned rc2 data'
print(json.dumps(verification, indent=2))
release = json.loads((RELEASE / 'release.json').read_text())
entries = json.loads((RELEASE / 'manifest.public.json').read_text())['bench']
by_id = {row['id']: row for row in entries}

## Distinguish entries, recordings, and resampling clusters

Identified duplicate/excerpt views remain in the legacy partition for historical continuity. A deterministic representative-only sensitivity view removes repeated views. Shared edits/extensions and known creators form dependence clusters; a fingerprint search is not proof of exhaustive uniqueness.

In [ ]:
print('partition                 entries   real     AI')
for partition in sorted(release['partitions']):
    selected = [r for r in entries if r['partition'] == partition]
    labels = Counter(r['label'] for r in selected)
    print(f"{partition:25s} {len(selected):7d} {labels['real']:6d} {labels['ai']:6d}")
print('Identified recording groups:', len({r['recording_group'] for r in entries}))
print('Dependence clusters:', len({r['dependence_cluster'] for r in entries}))
print('Representative entries:', sum(r['recording_representative'] for r in entries))
assert all(r['label'] == 'ai' for r in entries if r['partition'] != 'legacy')
print('Contemporary/native and demo partitions contain no real controls: independent contemporary FPR is undefined.')

In [ ]:
print('source                                      entries')
for source, count in sorted(Counter(r['source'] for r in entries).items()):
    print(f'{source:44s} {count:6d}')
print('Source cells are not necessarily independent generator families.')

## Contemporary evidence and verified native transport pairs

Suno's version is per-recording evidence. Udio's cohort is defined by creation date; its generator version remains unknown. Official demonstrations are provider-selected examples and must be reported separately. Only three MP3/AAC pairs passed full-waveform identity checks; seven MP3/MP3 candidates were rejected. Provider transport may change more than the codec.

In [ ]:
native = [r for r in entries if r['partition'] == 'contemporary_native']
for source in sorted({r['source'] for r in native}):
    selected = [r for r in native if r['source'] == source]
    print(source)
    print('  versions:', dict(Counter(r['generator_version'] or 'unknown' for r in selected)))
    print('  creator IDs:', len({r['creator_group_hash'] for r in selected}))
    print('  creation range:', min(r['created_at'] for r in selected), 'to', max(r['created_at'] for r in selected))
pairs = json.loads((RELEASE / 'native_pairs.public.json').read_text())
composition = Counter(by_id[p['primary_id']]['audio_codec'] + '/' + p['audio_codec'] for p in pairs)
assert sum(composition.values()) == release['native_transport_pairs']
print('Native transport pairs:', dict(composition))
assert all('waveform_identity' in p for p in pairs)
for pair in pairs:
    evidence = pair['waveform_identity']
    print(pair['primary_id'], 'full waveform correlation:', round(evidence['waveform_correlation'], 6))
print('Source identity evidence:', dict(Counter(r['source_identity_evidence'] for r in native)))

## Exposure is not the same as split-name disjointness

The native corpus was researcher-observed/mined before this evaluation. Exact path-stem checks against five local training/selection manifests are narrower than a full audio-level independence audit. External checkpoint membership is unknown where training manifests are unavailable.

In [ ]:
exposure = json.loads((RELEASE / 'exposure.public.json').read_text())
print('ArtifactNet:', exposure['artifactnet']['cohort_status'])
check = exposure['artifactnet']['local_manifest_check']
print('Local manifest checks:', len(check['input_hashes']))
print('Unresolved candidate collisions:', len(check['candidate_collisions']))
print('Check scope:', check['method'])
for name, status in sorted(exposure['external_checkpoints'].items()):
    print(name + ': ' + status['recording_membership'])
print(exposure['warning'])

## Next stages and limitations

Metadata verification is complete when the preceding cells pass. Audio reconstruction, full detector inference, transport inference, and result regeneration are separate checks; this notebook does not silently replace them with example scores. See `docs/PROTOCOL_v1.2.md`, `docs/DATASET_CARD_v1.2.md`, `docs/STATISTICS_v1.2.md`, and `docs/TRANSPORT_v1.2.md`, and `docs/UDIO_IDENTITY_CORRECTION_v1.2.md`.

This is a local preparation artifact, not evidence of a public release or arXiv submission. Metadata availability does not grant rights to redistribute audio, lyrics, prompts, or page snapshots.

Notebook execution follows the [official NBClient interface](https://nbclient.readthedocs.io/en/latest/client.html) and [Jupyter notebook format](https://nbformat.readthedocs.io/en/latest/format_description.html).